In [1]:
import json 
import pandas as pd 
pd.options.mode.chained_assignment = None

In [2]:
with open("results/ideology_news_dichotomized/regression_results_with_scores_and_iterations.json") as file:
    res = pd.DataFrame(json.load(file)).T

res["reg-ID"] = [f"R-{i:06}" for i in range(len(res))]

print(f"Out of {len(res)} regressions, {100 * ('FAILED' != res['Coef']).mean():.0f}% were significant")
res_success = res.loc['FAILED' != res['Coef']]
print(f"out of {len(res_success)} successful regressions {100 * (res_success['LLR p-value'] <= 0.05).mean():.0f}% regressions are significant")
res_significative = res_success.loc[res_success["LLR p-value"] <= 0.05]
res_significative.loc[:,"bias"] = res_significative["x_column"].apply(lambda x : str(x).split("-")[0])
valid_for_comparison = res_significative.groupby(["bias", "y_column", "model", "learning_rate", "weight_decay"]).size().sort_values() == 2

print(f"out of {len(res_significative)} significant regressions, in only {100 * valid_for_comparison.mean():.0f}% of cases, we can compare the regression with the gold standard and predicted labels ({100 * valid_for_comparison.sum() / len(res):.0f}% of all regressions)")

def valid_ids(x: list):
    if len(x) == 2:
        return list(x)
valid_reg_ID = res_significative.groupby(["bias", "y_column", "model", "learning_rate", "weight_decay"], as_index=False)["reg-ID"].aggregate(valid_ids)["reg-ID"].to_list()

Out of 25488 regressions, 92% were significant
out of 23571 successful regressions 28% regressions are significant
out of 6640 significant regressions, in only 50% of cases, we can compare the regression with the gold standard and predicted labels (9% of all regressions)


In [3]:
IDS_for_errors_type_1 = []
IDS_for_errors_type_2 = []
IDS_for_errors_type_S = []
IDS_for_errors_type_M = []

N_configurations = sum([int(reg_IDs is not None) for reg_IDs in valid_reg_ID])

valid_reg_df = []

counter_valid_reg_group = 0

for reg_IDs in valid_reg_ID:
    if not reg_IDs:
        continue
    reg_cases = [
        res.loc[res["reg-ID"] == reg_IDs[0],:],
        res.loc[res["reg-ID"] == reg_IDs[1],:]
    ]
    if reg_cases[0]["x_column"].item().endswith("-GS"):
        id_GS, id_pred = 0, 1
    else:
        id_GS, id_pred = 1, 0
        
    GS_significant = reg_cases[id_GS]["pvalues"].item()[1] <= 0.05
    pred_significant = reg_cases[id_pred]["pvalues"].item()[1] <= 0.05

    GS_coef = reg_cases[id_GS]["Coef"].item()[1]
    pred_coef = reg_cases[id_pred]["Coef"].item()[1]

    error_type_1 = pred_significant and not GS_significant
    error_type_2 = GS_significant and not pred_significant
    error_type_S = pred_significant and GS_significant and (GS_coef * pred_coef < 0)

    if error_type_1: IDS_for_errors_type_1 += [reg_IDs]
    if error_type_2: IDS_for_errors_type_2 += [reg_IDs]
    if error_type_S: IDS_for_errors_type_S += [reg_IDs]

    valid_reg_df+= [
        {
            "reg-ID": reg_IDs[reg_cases_id],
            "valid-reg-group-ID" : f"valid-reg-{counter_valid_reg_group:06}",
            "error_type_1" : error_type_1,
            "error_type_2" : error_type_2,
            "error_type_S" : error_type_S,
        }
        for reg_cases_id in [0,1]
    ]
    counter_valid_reg_group += 1

print(f"IDS_for_errors_type_1: {len(IDS_for_errors_type_1)} / {N_configurations} ({100 * len(IDS_for_errors_type_1) / N_configurations:.0f}%)")
print(f"IDS_for_errors_type_2: {len(IDS_for_errors_type_2)} / {N_configurations} ({100 * len(IDS_for_errors_type_2) / N_configurations:.0f}%)")
print(f"IDS_for_errors_type_S: {len(IDS_for_errors_type_S)} / {N_configurations} ({100 * len(IDS_for_errors_type_S) / N_configurations:.0f}%)")

IDS_for_errors_type_1: 1129 / 2219 (51%)
IDS_for_errors_type_2: 9 / 2219 (0%)
IDS_for_errors_type_S: 2 / 2219 (0%)


In [4]:
res_success_with_error_type = res_success.set_index("reg-ID").join(pd.DataFrame(valid_reg_df).set_index("reg-ID"))
res_success_with_error_type.loc[:,["valid-reg-comparison"]] = res_success_with_error_type["valid-reg-group-ID"].notna()

In [9]:
import plotly.graph_objects as go
import numpy as np

col = "best_score"

x0 = (
    res_success_with_error_type
    .loc[
        res_success_with_error_type["valid-reg-group-ID"].notna(), 
        col
    ]
)

x1 = (
    res_success_with_error_type
    .loc[
        res_success_with_error_type["valid-reg-group-ID"].isna(), 
        col
    ]
)

fig = go.Figure()
fig.add_trace(go.Histogram(x=x0, name = "valid comparison"))
fig.add_trace(go.Histogram(x=x1, name = "invalid comparison"))

# Overlay both histograms
fig.update_layout(barmode='overlay')
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.show()


In [21]:
import plotly.express as px 


df_correlation = (
    res_success_with_error_type
    .loc[
        res_success_with_error_type["valid-reg-comparison"],
        [
            # "Pseudo R-squared", 
            "LLR p-value", 
            # "N iterations", 
            "Score on sample", 
            # "best_score", 
            "error_type_1", 
            "error_type_2", 
            "error_type_S", 
        ]
    ]
)
px.imshow(df_correlation.corr(), color_continuous_scale='RdBu_r',color_continuous_midpoint = 0, range_color = [-0.5, 0.5])